<a href="https://colab.research.google.com/github/dehigher/ai-practice/blob/main/3_%E5%88%9D%E8%AF%86LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip3 install langchain

### 提示词模版

In [3]:
!pip3 install langchain-deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.3 MB/s eta 0:00:00


In [15]:
import os
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

os.environ["DEEPSEEK_API_KEY"] = "sk-3547bbf4b0da4f8eb5a99a049e2614d4"

model = init_chat_model(
    model="deepseek-chat",
    api_base="https://api.deepseek.com"
)

# 构造模版提示词
prompt_template = "What is a good name for a company that makes {product}? And only return the best one."
prompt = PromptTemplate.from_template(prompt_template)

# 输入参数，单个调用
singleInput = prompt.format(product='colorful socks')
resp = model.invoke(singleInput)
print("single:{}".format(resp.content))


# 输入参数，批量调用
products = [
    "cloudnative devops platform",
    "Noise cancellation headphone",
    "colorful socks",
]
for p in products:
  batchInput = prompt.format(product=p)
  resp = model.invoke(batchInput)
  print("batchInput:{}".format(resp.content))

single:ChromaSox
batchInput:**Nexus**
batchInput:Aether
batchInput:ChromaToes


## HTTP request chain

In [4]:
!pip install langchain-deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.0 MB/s eta 0:00:00


In [16]:
import os
import requests
from textwrap import dedent
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

os.environ["DEEPSEEK_API_KEY"] = "sk-3547bbf4b0da4f8eb5a99a049e2614d4"



model = init_chat_model(
    model="deepseek-chat",
    api_base="https://api.deepseek.com"
)

# 构造模版提示词，dedent格式化字符串，对齐作用
template = dedent(
    """
      Between >>> and <<< are the raw search result text from web.
      Extract the answer to the question '{query}' or say "not found" if the information is not contained.
      Use the format
      Extracted:<answer or "not found">
      >>> {requests_result} <<<
      Extracted:
    """
)


prompt = PromptTemplate(
    input_variables = ["query", "requests_result" ],
    template = template
)

def query_build(question):
  question = question.replace(" ", "+")
  url = "http://www.baidu.com/s?wd=" + question
  resp = requests.get(url=url, timeout=10)
  resp.raise_for_status() # 检查响应状态码抛异常
  search_text = resp.text[:8000] #截断一下避免太长

  result = model.invoke(prompt.format(
      query = question,
      requests_result = search_text
  ))

  # result 可能是 BaseMessage / str，做一下兼容
  if hasattr(result, 'content'):
    return result.content
  return result


print(query_build("今天天气怎样"))

Extracted: not found


## 基于langchain实现调用链

In [19]:
!pip install -U langchain

In [25]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence  # 或直接用 >> 运算符


os.environ["DEEPSEEK_API_KEY"] = "sk-3547bbf4b0da4f8eb5a99a049e2614d4"
model = init_chat_model(
    model="deepseek-chat",
    api_base="https://api.deepseek.com"
)


prompt1 = ChatPromptTemplate.from_template("给下面问题写一个简短回答：{question}")
prompt2 = ChatPromptTemplate.from_template("把下面回答翻译成英文：{answer}")

# 定义第一个子链：问题 -> 中文回答
chain1 = prompt1 | model

# 定义第二个子链：中文回答 -> 英文回答
chain2 = prompt2 | model

# 使用 RunnableSequence 显式串联
full_chain = RunnableSequence(
    first=chain1,
    middle=[],
    last=chain2
)

res = full_chain.invoke({"question": "今天天气怎么样？"})
print(res)

content='Since I cannot access real-time data, to check today\'s weather, you can:  \n\n1. **Use a weather app** (such as your phone’s built-in app or apps like Moji Weather)  \n2. **Search "weather + your city name"** online  \n3. **Ask a smart device** (like Xiao Ai, Siri, etc.)  \n4. **Look outside the window** (the most direct way!)  \n\nIf you tell me your city, I can help you look up the latest weather forecast. 🌤️' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 390, 'total_tokens': 503, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 390}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'a69e8044-9b0f-44ce-aad6-a11c5191bec4', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cb8de-0b71-7213-a2d8-deb364fbfcae-0' tool